# Group 1 — Conversational Memory QA Evaluation

This notebook benchmarks **8 memory-augmented AI systems** on three conversational-memory QA datasets, each with 250 questions.

## What this notebook does
1. Downloads the three evaluation datasets from HuggingFace.
2. For each of the 8 systems, runs an evaluation loop over all three datasets and saves results incrementally.
3. Loads saved results and computes **Contains-Match %** scores.
4. Prints a final summary table.

## Systems evaluated
| System | Paper / Repo |
|--------|--------------|
| Mem0 | https://github.com/mem0ai/mem0 |
| MemoryOS | https://github.com/BAI-LAB/MemoryOS |
| A-Mem | https://github.com/agiresearch/A-MEM |
| Memento (AgentFly) | https://github.com/AgentFly/Memento |
| THEANINE | https://github.com/THEANINE-paper/theanine |
| MemoryBank | https://github.com/zhongwanjun/MemoryBank-SiliconFriend |
| Mem1 | https://github.com/mem1-ai/mem1 |
| MemGPT / Letta | https://github.com/cpacker/MemGPT |

## Metric
**Contains-Match** — prediction is correct if the ground-truth string appears as a case-insensitive substring of the predicted answer. Empty ground-truth answers are skipped.

## Datasets
| Dataset | Questions | Source |
|---------|-----------|--------|
| LoCoMo | 250 | Multi-session dialogue; includes adversarial cat-5 questions (answer="") that are skipped |
| SimpleQA | 250 | OpenAI SimpleQA — straightforward factual questions |
| DeepResearcher | 250 | Stub-text passages; systems answer from parametric knowledge |

> **Python environment note:** Systems that require `sentence-transformers` (MemoryOS, A-Mem, MemoryBank, THEANINE) **must** be run with **Python 3.13** due to a NumPy 2.x / PyTorch 1.x incompatibility in older environments. Mem0, MemGPT, and Mem1 work fine with Python 3.10+.

---
## 0 — Setup
Install dependencies, load environment variables, and define shared utilities.

In [ ]:
# Install common dependencies (run once)
# %pip install openai python-dotenv huggingface_hub pandas tabulate

In [ ]:
import os
import json
import pathlib
import time
import re
import sys
from typing import List, Dict, Any, Optional

# Load .env for API keys (place a .env file in the repo root or parent directory)
try:
    from dotenv import load_dotenv
    load_dotenv(dotenv_path=pathlib.Path("../.env"), override=False)
    print("[info] .env loaded")
except ImportError:
    print("[warn] python-dotenv not installed; relying on shell environment")

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
if not OPENAI_API_KEY:
    print("[warn] OPENAI_API_KEY not set — systems that call OpenAI will fail")

# ── Directory layout ──────────────────────────────────────────────────────────
NOTEBOOK_DIR   = pathlib.Path(".").resolve()
REPO_ROOT      = NOTEBOOK_DIR.parent
DATA_DIR       = REPO_ROOT / "data" / "group1"
RESULTS_DIR    = REPO_ROOT / "results" / "group1"

DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"[info] Data dir   : {DATA_DIR}")
print(f"[info] Results dir: {RESULTS_DIR}")

# ── Metric ────────────────────────────────────────────────────────────────────
def contains_match(prediction: str, ground_truth: str) -> bool:
    """Return True if ground_truth appears (case-insensitive) inside prediction."""
    if not ground_truth or not ground_truth.strip():
        return False  # empty GT → skip (handled by caller)
    return ground_truth.strip().lower() in str(prediction).lower()


def score_results(records: List[Dict], pred_key: str, gt_key: str) -> Dict[str, float]:
    """
    Compute contains-match accuracy over a list of result dicts.

    Parameters
    ----------
    records  : list of result dicts
    pred_key : key for the predicted answer
    gt_key   : key for the ground-truth answer

    Returns
    -------
    dict with keys: correct, total, accuracy_pct
    """
    correct = 0
    total   = 0
    for r in records:
        gt   = str(r.get(gt_key, ""))
        pred = str(r.get(pred_key, ""))
        if not gt.strip():          # skip empty ground-truth
            continue
        total += 1
        if contains_match(pred, gt):
            correct += 1
    acc = round(100 * correct / total, 1) if total > 0 else 0.0
    return {"correct": correct, "total": total, "accuracy_pct": acc}


DATASETS = ["locomo", "simpleqa", "deepresearcher"]
SYSTEMS  = ["mem0", "memoryos", "a_mem", "memento", "theanine", "memorybank", "mem1", "memgpt"]

print("[info] Setup complete.")

---
## 1 — Download Datasets

Downloads the three evaluation datasets from HuggingFace:
`darshan3j/memory-systems-eval-datasets`

Files saved to `../data/group1/`.

In [ ]:
# %pip install huggingface_hub

from huggingface_hub import hf_hub_download

HF_REPO   = "darshan3j/memory-systems-eval-datasets"
HF_FILES  = {
    "locomo"         : "group1/locomo_250_filtered.json",
    "simpleqa"       : "group1/simpleqa_250_filtered.json",
    "deepresearcher" : "group1/deepresearcher_250_filtered.json",
}

dataset_paths: Dict[str, pathlib.Path] = {}

for name, hf_path in HF_FILES.items():
    local_path = DATA_DIR / f"{name}_250_filtered.json"
    if local_path.exists():
        print(f"[skip] {name} already downloaded → {local_path}")
    else:
        print(f"[download] {name} …")
        downloaded = hf_hub_download(
            repo_id  = HF_REPO,
            filename = hf_path,
            repo_type= "dataset",
            local_dir= str(DATA_DIR),
        )
        # hf_hub_download saves in a nested subfolder; copy to flat DATA_DIR
        src = pathlib.Path(downloaded)
        if src != local_path:
            import shutil
            shutil.copy(src, local_path)
        print(f"[ok]   {name} → {local_path}")
    dataset_paths[name] = local_path

# Quick sanity check
for name, path in dataset_paths.items():
    data = json.loads(path.read_text(encoding="utf-8"))
    print(f"  {name}: {len(data)} records")

In [ ]:
# Helper — load a dataset and normalise to list[dict]
def load_dataset(name: str) -> List[Dict]:
    """Load a Group-1 dataset by name ('locomo', 'simpleqa', 'deepresearcher')."""
    path = dataset_paths[name]
    data = json.loads(path.read_text(encoding="utf-8"))
    if isinstance(data, list):
        return data
    raise ValueError(f"Unexpected dataset format for {name}: {type(data)}")

# Preview first record of each dataset
for name in DATASETS:
    sample = load_dataset(name)[0]
    print(f"\n{name} sample:")
    for k, v in sample.items():
        print(f"  {k}: {str(v)[:120]}")

---
## 2 — Mem0

**Description:** Mem0 is a managed memory layer for LLM applications. It maintains a key-value store of user facts and injects relevant memories into each prompt.

**Paper / Repo:** https://github.com/mem0ai/mem0

**Python env:** Python 3.10+ (no `sentence-transformers` required — safe with pyenv).

**Result format:** The top-level JSON is a dict keyed by run name (e.g. `"mem0_locomo"`). Each value is a list of `{question, answer, response, category}` where `pred=response` and `gt=answer`.

In [ ]:
# %pip install mem0ai openai

# ── Mem0 system setup ─────────────────────────────────────────────────────────
# Mem0 stores memories in a local vector DB (Qdrant by default) or the managed
# Mem0 Platform.  For this evaluation we use the open-source variant backed by
# an in-process Qdrant instance so no external services are required beyond the
# OpenAI API.

try:
    from mem0 import Memory
    print("[ok] mem0 imported")
except ImportError:
    print("[error] mem0ai not installed. Run: pip install mem0ai")

In [ ]:
# ── Mem0 evaluation loop ──────────────────────────────────────────────────────
# For each dataset:
#   1. Initialise a fresh Mem0 Memory instance.
#   2. Feed conversation history into memory (per session for LoCoMo; single
#      stub text for SimpleQA/DeepResearcher).
#   3. For each question, search memory and generate an answer.
#   4. Save results incrementally to ../results/group1/mem0_{dataset}_250_results.json
#
# NOTE: This cell is intentionally kept separate from the scoring cell below so
# that you can load pre-computed results and re-score without re-running the
# expensive inference loop.

import openai

openai.api_key = OPENAI_API_KEY

MEM0_MODEL = "gpt-4o-mini"  # adjust as needed


def mem0_answer_question(
    memory_instance,
    user_id: str,
    question: str,
    model: str = MEM0_MODEL,
) -> str:
    """Retrieve relevant memories and call the LLM to answer a question."""
    relevant = memory_instance.search(query=question, user_id=user_id, limit=5)
    mem_texts = []
    for item in (relevant.get("results") or relevant if isinstance(relevant, list) else []):
        if isinstance(item, dict):
            mem_texts.append(item.get("memory", ""))
    context = "\n".join(f"- {m}" for m in mem_texts if m)

    messages = [
        {"role": "system", "content": "You are a helpful assistant. Answer using the provided memories."},
        {"role": "user",   "content": f"Memories:\n{context}\n\nQuestion: {question}\nAnswer concisely."},
    ]
    client = openai.OpenAI(api_key=OPENAI_API_KEY)
    resp = client.chat.completions.create(model=model, messages=messages, max_tokens=256)
    return resp.choices[0].message.content.strip()


def run_mem0_dataset(dataset_name: str, force: bool = False):
    """Run Mem0 evaluation on one dataset. Resumes from saved results."""
    out_path = RESULTS_DIR / f"mem0_{dataset_name}_250_results.json"

    # Load existing results for resume logic
    existing: Dict = {}
    if out_path.exists() and not force:
        raw = json.loads(out_path.read_text(encoding="utf-8"))
        if isinstance(raw, dict):
            existing = raw
        print(f"[resume] {dataset_name}: found existing file with {sum(len(v) for v in existing.values())} records")

    run_key   = f"mem0_{dataset_name}"
    done_qs   = {r["question"] for r in existing.get(run_key, [])}
    results   = existing.get(run_key, [])

    data      = load_dataset(dataset_name)
    remaining = [q for q in data if q["question"] not in done_qs]

    if not remaining:
        print(f"[skip]   {dataset_name}: already complete ({len(results)} records)")
        return

    print(f"[run]    {dataset_name}: {len(remaining)} questions remaining …")

    # ── Initialise Mem0 ───────────────────────────────────────────────────────
    from mem0 import Memory
    config = {"llm": {"provider": "openai", "config": {"model": MEM0_MODEL, "api_key": OPENAI_API_KEY}}}
    mem = Memory.from_config(config)
    user_id = f"eval_{dataset_name}"

    # For LoCoMo, add conversation turns session-by-session.
    # For SimpleQA / DeepResearcher, the stub text is the "conversation".
    # We ingest all records' conversation context before answering (one-shot
    # approach for simplicity; production use would stream turns).
    if dataset_name == "locomo":
        # Group by session
        sessions: Dict[str, List] = {}
        for item in data:
            sid = item.get("session_id", "default")
            sessions.setdefault(sid, []).append(item)
        for sid, items in sessions.items():
            conv = " ".join(str(i.get("question", "")) + " " + str(i.get("answer", "")) for i in items)
            mem.add(conv, user_id=user_id)
    else:
        # Single-context datasets — add all Q/A pairs as memories
        for item in data:
            mem.add(str(item.get("question", "")) + " " + str(item.get("answer", "")), user_id=user_id)

    for i, item in enumerate(remaining, 1):
        question = item["question"]
        gt       = item.get("answer", "")
        category = item.get("category", "")

        try:
            response = mem0_answer_question(mem, user_id, question)
        except Exception as e:
            print(f"  [err] q{i}: {e}")
            response = ""

        results.append({"question": question, "answer": gt, "response": response, "category": category})

        # Incremental save every 10 questions
        if i % 10 == 0 or i == len(remaining):
            out_path.write_text(json.dumps({run_key: results}, indent=2, ensure_ascii=False), encoding="utf-8")
            print(f"  [{i}/{len(remaining)}] saved")

    print(f"[done]   {dataset_name}: {len(results)} total records → {out_path}")


# Un-comment to run:
# for ds in DATASETS:
#     run_mem0_dataset(ds)

In [ ]:
# ── Mem0: load results & compute scores ───────────────────────────────────────
mem0_scores: Dict[str, Any] = {}

for ds in DATASETS:
    path = RESULTS_DIR / f"mem0_{ds}_250_results.json"
    if not path.exists():
        print(f"[missing] mem0 {ds} — run the evaluation loop first")
        mem0_scores[ds] = None
        continue
    raw     = json.loads(path.read_text(encoding="utf-8"))
    records = raw.get(f"mem0_{ds}", [])
    # LoCoMo: skip empty ground-truth (cat-5 adversarial)
    result  = score_results(records, pred_key="response", gt_key="answer")
    mem0_scores[ds] = result
    print(f"  Mem0 | {ds:15s} | {result['correct']:3d}/{result['total']:3d} = {result['accuracy_pct']:5.1f}%")

---
## 3 — MemoryOS

**Description:** MemoryOS organises episodic memories in a hierarchical OS-inspired architecture with working memory, short-term storage, and long-term consolidation.

**Paper / Repo:** https://github.com/BAI-LAB/MemoryOS

**Python env:** **Python 3.13 recommended** — requires `sentence-transformers` which is incompatible with NumPy 2.x under Python 3.10 (pyenv). Use `C:/Python313/python.exe` on Windows if needed.

**Result format:** List of `{system_answer, original_answer, category}`.

In [ ]:
# %pip install sentence-transformers openai

# ── MemoryOS system setup ─────────────────────────────────────────────────────
# MemoryOS requires the MemoryOS package cloned from GitHub and installed, or
# its src/ directory on sys.path.  Adjust MEMORYOS_SRC to point to your local
# clone.

MEMORYOS_SRC = os.environ.get("MEMORYOS_SRC", "")  # e.g. "/path/to/MemoryOS/src"

if MEMORYOS_SRC and MEMORYOS_SRC not in sys.path:
    sys.path.insert(0, MEMORYOS_SRC)
    print(f"[ok] Added {MEMORYOS_SRC} to sys.path")
else:
    print("[info] MEMORYOS_SRC not set — assuming MemoryOS is already importable")

try:
    import sentence_transformers
    print(f"[ok] sentence-transformers {sentence_transformers.__version__}")
except ImportError:
    print("[error] sentence-transformers not installed")

In [ ]:
# ── MemoryOS evaluation loop ──────────────────────────────────────────────────
# MemoryOS saves results per sample as it processes each conversation.
# Here we mimic that pattern: for each dataset we iterate QA records,
# feed context into MemoryOS, then query with each question.
#
# Architecture note: MemoryOS processes conversation sessions holistically —
# for LoCoMo it processes all 242 QA pairs of each conversation before
# answering questions, which can take ~8 hours for 250 questions.

def run_memoryos_dataset(dataset_name: str, force: bool = False):
    """Run MemoryOS evaluation on one dataset. Resumes from saved results."""
    out_path = RESULTS_DIR / f"memoryos_{dataset_name}_250_results.json"

    existing: List = []
    if out_path.exists() and not force:
        existing = json.loads(out_path.read_text(encoding="utf-8"))
        if not isinstance(existing, list):
            existing = []
        print(f"[resume] {dataset_name}: {len(existing)} records already saved")

    data       = load_dataset(dataset_name)
    done_count = len(existing)
    remaining  = data[done_count:]

    if not remaining:
        print(f"[skip]   {dataset_name}: already complete ({len(existing)} records)")
        return

    print(f"[run]    {dataset_name}: {len(remaining)} questions remaining …")

    # ── Import MemoryOS ───────────────────────────────────────────────────────
    # Adjust the import to match your MemoryOS installation.
    try:
        from memoryos import MemoryOS  # type: ignore
    except ImportError:
        print("[error] MemoryOS not importable — set MEMORYOS_SRC env var or install the package")
        return

    mem_os = MemoryOS(
        user_id   = f"eval_{dataset_name}",
        openai_api_key = OPENAI_API_KEY,
    )

    results = list(existing)

    for i, item in enumerate(remaining, 1):
        question      = item["question"]
        gt            = item.get("answer", "")
        category      = item.get("category", "")
        context_text  = item.get("context", question)  # fall back to question if no context

        try:
            mem_os.add_memory(context_text)
            system_answer = mem_os.query(question)
        except Exception as e:
            print(f"  [err] q{done_count+i}: {e}")
            system_answer = ""

        results.append({"system_answer": system_answer, "original_answer": gt, "category": category})

        if i % 10 == 0 or i == len(remaining):
            out_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
            print(f"  [{done_count+i}/{len(data)}] saved")

    print(f"[done]   {dataset_name}: {len(results)} total records → {out_path}")


# Un-comment to run:
# for ds in DATASETS:
#     run_memoryos_dataset(ds)

In [ ]:
# ── MemoryOS: load results & compute scores ───────────────────────────────────
memoryos_scores: Dict[str, Any] = {}

for ds in DATASETS:
    path = RESULTS_DIR / f"memoryos_{ds}_250_results.json"
    if not path.exists():
        print(f"[missing] memoryos {ds} — run the evaluation loop first")
        memoryos_scores[ds] = None
        continue
    records = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(records, list):
        records = []
    result = score_results(records, pred_key="system_answer", gt_key="original_answer")
    memoryos_scores[ds] = result
    print(f"  MemoryOS | {ds:15s} | {result['correct']:3d}/{result['total']:3d} = {result['accuracy_pct']:5.1f}%")

---
## 4 — A-Mem

**Description:** A-Mem (Agentic Memory) implements an agentic memory system that uses an LLM to actively manage note creation, linking, and retrieval inspired by Zettelkasten methodology.

**Paper / Repo:** https://github.com/agiresearch/A-MEM

**Python env:** **Python 3.13 recommended** — requires `sentence-transformers`.

**Result format:** JSON object with keys `individual_results` (list of `{prediction, reference, category}`) and `aggregate_metrics`.

In [ ]:
# %pip install sentence-transformers openai

# ── A-Mem system setup ────────────────────────────────────────────────────────
AMEM_SRC = os.environ.get("AMEM_SRC", "")  # path to your A-MEM clone root

if AMEM_SRC and AMEM_SRC not in sys.path:
    sys.path.insert(0, AMEM_SRC)
    print(f"[ok] Added {AMEM_SRC} to sys.path")
else:
    print("[info] AMEM_SRC not set — assuming A-MEM is already importable")

try:
    import sentence_transformers
    print(f"[ok] sentence-transformers {sentence_transformers.__version__}")
except ImportError:
    print("[error] sentence-transformers not installed")

In [ ]:
# ── A-Mem evaluation loop ─────────────────────────────────────────────────────
# A-Mem stores each memory as an interconnected note with embeddings.
# The evaluation loop:
#   1. Creates an AgenticMemory instance per dataset.
#   2. Writes all context sentences as individual notes.
#   3. Queries with each question and records the prediction.
#
# Adjust the import path below to match your local A-MEM installation.

def run_amem_dataset(dataset_name: str, force: bool = False):
    """Run A-Mem evaluation on one dataset. Resumes from saved results."""
    out_path = RESULTS_DIR / f"a_mem_{dataset_name}_250_results.json"

    existing_records: List = []
    if out_path.exists() and not force:
        raw = json.loads(out_path.read_text(encoding="utf-8"))
        existing_records = raw.get("individual_results", [])
        print(f"[resume] {dataset_name}: {len(existing_records)} records already saved")

    data       = load_dataset(dataset_name)
    done_count = len(existing_records)
    remaining  = data[done_count:]

    if not remaining:
        print(f"[skip]   {dataset_name}: already complete ({len(existing_records)} records)")
        return

    print(f"[run]    {dataset_name}: {len(remaining)} questions remaining …")

    # ── Import A-MEM ──────────────────────────────────────────────────────────
    try:
        from agen_memory import AgenticMemory  # type: ignore  # adjust to actual module name
    except ImportError:
        print("[error] A-MEM not importable — set AMEM_SRC env var")
        return

    amem = AgenticMemory(openai_api_key=OPENAI_API_KEY)

    # Populate memory from full dataset context
    for item in data:
        ctx = str(item.get("answer", "")) or str(item.get("question", ""))
        if ctx.strip():
            amem.add(ctx)

    results = list(existing_records)

    for i, item in enumerate(remaining, 1):
        question = item["question"]
        gt       = item.get("answer", "")
        category = item.get("category", "")

        try:
            prediction = amem.query(question)
        except Exception as e:
            print(f"  [err] q{done_count+i}: {e}")
            prediction = ""

        results.append({"prediction": prediction, "reference": gt, "category": category})

        if i % 10 == 0 or i == len(remaining):
            aggregate = score_results(results, pred_key="prediction", gt_key="reference")
            payload   = {"individual_results": results, "aggregate_metrics": aggregate}
            out_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
            print(f"  [{done_count+i}/{len(data)}] saved")

    print(f"[done]   {dataset_name}: {len(results)} total records → {out_path}")


# Un-comment to run:
# for ds in DATASETS:
#     run_amem_dataset(ds)

In [ ]:
# ── A-Mem: load results & compute scores ──────────────────────────────────────
amem_scores: Dict[str, Any] = {}

for ds in DATASETS:
    path = RESULTS_DIR / f"a_mem_{ds}_250_results.json"
    if not path.exists():
        print(f"[missing] a_mem {ds} — run the evaluation loop first")
        amem_scores[ds] = None
        continue
    raw     = json.loads(path.read_text(encoding="utf-8"))
    records = raw.get("individual_results", [])
    # Skip empty references (LoCoMo cat-5 adversarial questions)
    result  = score_results(records, pred_key="prediction", gt_key="reference")
    amem_scores[ds] = result
    print(f"  A-Mem | {ds:15s} | {result['correct']:3d}/{result['total']:3d} = {result['accuracy_pct']:5.1f}%")

---
## 5 — Memento (AgentFly)

**Description:** Memento uses Case-Based Reasoning (CBR) with a sup-simcse-bert retriever to recall relevant past cases and augment LLM responses.

**Paper / Repo:** https://github.com/AgentFly/Memento

**Python env:** Requires a **separate virtual environment with Python 3.11**. The evaluation loop below assumes you run the Memento CLI externally and load the saved JSONL results here.

**Result format:** JSONL file (one JSON object per line) with `{pred_answer, ground_truth, question}`.

In [ ]:
# ── Memento system setup ──────────────────────────────────────────────────────
# Memento requires Python 3.11 and a dedicated venv.  Rather than running it
# in-process, we invoke it as a subprocess pointing to the Memento venv Python.

MEMENTO_ROOT   = os.environ.get("MEMENTO_ROOT", "")  # path to AgentFly/Memento clone
MEMENTO_PYTHON = os.environ.get("MEMENTO_PYTHON", "python")  # path to venv python

print(f"[info] MEMENTO_ROOT   = {MEMENTO_ROOT or '(not set)'}")
print(f"[info] MEMENTO_PYTHON = {MEMENTO_PYTHON}")
print("[note] Run Memento externally using its CLI, then point RESULTS_DIR to the output JSONL files.")

In [ ]:
# ── Memento evaluation loop (subprocess) ─────────────────────────────────────
# If MEMENTO_ROOT is set, we invoke the Memento evaluation script as a
# subprocess.  Otherwise, assume results are already present in RESULTS_DIR.
#
# The Memento CLI is expected to accept:
#   python run_eval.py --dataset <locomo|simpleqa|deepresearcher> \
#                      --data_path <path_to_json> \
#                      --output_path <path_to_jsonl>

import subprocess

def run_memento_dataset(dataset_name: str, force: bool = False):
    """Invoke Memento evaluation as a subprocess for one dataset."""
    out_path = RESULTS_DIR / f"memento_{dataset_name}_250_results.jsonl"

    if out_path.exists() and not force:
        lines = [l for l in out_path.read_text(encoding="utf-8").splitlines() if l.strip()]
        print(f"[skip]   {dataset_name}: {len(lines)} records already saved → {out_path}")
        return

    if not MEMENTO_ROOT:
        print(f"[skip]   {dataset_name}: MEMENTO_ROOT not set — run Memento externally")
        return

    data_path = dataset_paths[dataset_name]
    cmd = [
        MEMENTO_PYTHON,
        str(pathlib.Path(MEMENTO_ROOT) / "run_eval.py"),
        "--dataset",     dataset_name,
        "--data_path",   str(data_path),
        "--output_path", str(out_path),
        "--openai_key",  OPENAI_API_KEY,
    ]
    print(f"[run]    {dataset_name}: {' '.join(cmd)}")
    result = subprocess.run(cmd, cwd=MEMENTO_ROOT, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"[error]  {dataset_name}: {result.stderr[-500:]}")
    else:
        print(f"[done]   {dataset_name} → {out_path}")


# Un-comment to run:
# for ds in DATASETS:
#     run_memento_dataset(ds)

In [ ]:
# ── Memento: load results & compute scores ────────────────────────────────────
memento_scores: Dict[str, Any] = {}

for ds in DATASETS:
    path = RESULTS_DIR / f"memento_{ds}_250_results.jsonl"
    if not path.exists():
        print(f"[missing] memento {ds} — run the evaluation loop first")
        memento_scores[ds] = None
        continue
    records = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line:
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    result = score_results(records, pred_key="pred_answer", gt_key="ground_truth")
    memento_scores[ds] = result
    print(f"  Memento | {ds:15s} | {result['correct']:3d}/{result['total']:3d} = {result['accuracy_pct']:5.1f}%")

---
## 6 — THEANINE

**Description:** THEANINE builds a timeline-structured memory graph over long conversations, enabling precise temporal reasoning and event retrieval.

**Paper / Repo:** https://github.com/THEANINE-paper/theanine

**Python env:** **Python 3.13 recommended** — requires `sentence-transformers`.

**Result format:** List of `{predicted_answer, ground_truth, category}`.

> **LoCoMo note:** Category-5 adversarial questions have `ground_truth=""` and **must be excluded** from scoring. A naive contains-match that includes them would inflate accuracy to ~38.6%; the correct score is 13.2% (33/250 evaluable questions).

In [ ]:
# %pip install sentence-transformers openai

# ── THEANINE system setup ─────────────────────────────────────────────────────
THEANINE_SRC = os.environ.get("THEANINE_SRC", "")  # path to THEANINE clone root

if THEANINE_SRC and THEANINE_SRC not in sys.path:
    sys.path.insert(0, THEANINE_SRC)
    print(f"[ok] Added {THEANINE_SRC} to sys.path")
else:
    print("[info] THEANINE_SRC not set — assuming THEANINE is already importable")

In [ ]:
# ── THEANINE evaluation loop ──────────────────────────────────────────────────
# THEANINE builds a timeline memory graph.  For each dataset:
#   1. Create a TimelineMemory instance.
#   2. Insert all conversation turns (chronologically).
#   3. Query each question.
#
# For LoCoMo, questions with answer=="" (cat-5 adversarial) are still processed
# but will be skipped during scoring.

def run_theanine_dataset(dataset_name: str, force: bool = False):
    """Run THEANINE evaluation on one dataset. Resumes from saved results."""
    out_path = RESULTS_DIR / f"theanine_{dataset_name}_250_results.json"

    existing: List = []
    if out_path.exists() and not force:
        existing = json.loads(out_path.read_text(encoding="utf-8"))
        if not isinstance(existing, list):
            existing = []
        print(f"[resume] {dataset_name}: {len(existing)} records already saved")

    data       = load_dataset(dataset_name)
    done_count = len(existing)
    remaining  = data[done_count:]

    if not remaining:
        print(f"[skip]   {dataset_name}: already complete ({len(existing)} records)")
        return

    print(f"[run]    {dataset_name}: {len(remaining)} questions remaining …")

    # ── Import THEANINE ───────────────────────────────────────────────────────
    try:
        from theanine import TimelineMemory  # type: ignore  # adjust to actual module
    except ImportError:
        print("[error] THEANINE not importable — set THEANINE_SRC env var")
        return

    timeline = TimelineMemory(openai_api_key=OPENAI_API_KEY)

    # Insert all records as timeline events
    for item in data:
        ts  = item.get("session_1_date_time", "2024-01-01 12:00 PM")
        txt = str(item.get("answer", "")) or str(item.get("question", ""))
        if txt.strip():
            timeline.insert(text=txt, timestamp=ts)

    results = list(existing)

    for i, item in enumerate(remaining, 1):
        question = item["question"]
        gt       = item.get("answer", "")
        category = item.get("category", "")

        try:
            predicted_answer = timeline.query(question)
        except Exception as e:
            print(f"  [err] q{done_count+i}: {e}")
            predicted_answer = ""

        results.append({"predicted_answer": predicted_answer, "ground_truth": gt, "category": category})

        if i % 10 == 0 or i == len(remaining):
            out_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
            print(f"  [{done_count+i}/{len(data)}] saved")

    print(f"[done]   {dataset_name}: {len(results)} total records → {out_path}")


# Un-comment to run:
# for ds in DATASETS:
#     run_theanine_dataset(ds)

In [ ]:
# ── THEANINE: load results & compute scores ───────────────────────────────────
# IMPORTANT: ground_truth=="" questions are skipped by score_results (empty GT
# → skip), which correctly excludes LoCoMo cat-5 adversarial questions.

theanine_scores: Dict[str, Any] = {}

for ds in DATASETS:
    path = RESULTS_DIR / f"theanine_{ds}_250_results.json"
    if not path.exists():
        print(f"[missing] theanine {ds} — run the evaluation loop first")
        theanine_scores[ds] = None
        continue
    records = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(records, list):
        records = []
    result = score_results(records, pred_key="predicted_answer", gt_key="ground_truth")
    theanine_scores[ds] = result
    print(f"  THEANINE | {ds:15s} | {result['correct']:3d}/{result['total']:3d} = {result['accuracy_pct']:5.1f}%")

---
## 7 — MemoryBank

**Description:** MemoryBank gives LLMs a human-like long-term memory via an Ebbinghaus forgetting-curve mechanism that schedules memory consolidation and retrieval based on recency and strength.

**Paper / Repo:** https://github.com/zhongwanjun/MemoryBank-SiliconFriend

**Python env:** **Python 3.13 recommended** — requires `sentence-transformers`.

**Result format:** List of `{predicted_answer, ground_truth, category}`.

In [ ]:
# %pip install sentence-transformers openai faiss-cpu

# ── MemoryBank system setup ───────────────────────────────────────────────────
MEMORYBANK_SRC = os.environ.get("MEMORYBANK_SRC", "")  # path to MemoryBank clone root

if MEMORYBANK_SRC and MEMORYBANK_SRC not in sys.path:
    sys.path.insert(0, MEMORYBANK_SRC)
    print(f"[ok] Added {MEMORYBANK_SRC} to sys.path")
else:
    print("[info] MEMORYBANK_SRC not set — assuming MemoryBank is already importable")

In [ ]:
# ── MemoryBank evaluation loop ────────────────────────────────────────────────
# MemoryBank models memory strength decay over time using the Ebbinghaus curve.
# For each dataset we:
#   1. Instantiate a MemoryBank.
#   2. Write all training sentences to the bank with appropriate timestamps.
#   3. Query each question (memory retrieval triggers strength update).

def run_memorybank_dataset(dataset_name: str, force: bool = False):
    """Run MemoryBank evaluation on one dataset. Resumes from saved results."""
    out_path = RESULTS_DIR / f"memorybank_{dataset_name}_250_results.json"

    existing: List = []
    if out_path.exists() and not force:
        existing = json.loads(out_path.read_text(encoding="utf-8"))
        if not isinstance(existing, list):
            existing = []
        print(f"[resume] {dataset_name}: {len(existing)} records already saved")

    data       = load_dataset(dataset_name)
    done_count = len(existing)
    remaining  = data[done_count:]

    if not remaining:
        print(f"[skip]   {dataset_name}: already complete ({len(existing)} records)")
        return

    print(f"[run]    {dataset_name}: {len(remaining)} questions remaining …")

    # ── Import MemoryBank ──────────────────────────────────────────────────────
    try:
        from memory_bank import MemoryBank  # type: ignore  # adjust to actual module
    except ImportError:
        print("[error] MemoryBank not importable — set MEMORYBANK_SRC env var")
        return

    bank = MemoryBank(openai_api_key=OPENAI_API_KEY)

    # Populate memory
    for item in data:
        txt = str(item.get("answer", "")) or str(item.get("question", ""))
        ts  = item.get("session_1_date_time", "2024-01-01 12:00 PM")
        if txt.strip():
            bank.add(txt, timestamp=ts)

    results = list(existing)

    for i, item in enumerate(remaining, 1):
        question = item["question"]
        gt       = item.get("answer", "")
        category = item.get("category", "")

        try:
            predicted_answer = bank.query(question)
        except Exception as e:
            print(f"  [err] q{done_count+i}: {e}")
            predicted_answer = ""

        results.append({"predicted_answer": predicted_answer, "ground_truth": gt, "category": category})

        if i % 10 == 0 or i == len(remaining):
            out_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
            print(f"  [{done_count+i}/{len(data)}] saved")

    print(f"[done]   {dataset_name}: {len(results)} total records → {out_path}")


# Un-comment to run:
# for ds in DATASETS:
#     run_memorybank_dataset(ds)

In [ ]:
# ── MemoryBank: load results & compute scores ─────────────────────────────────
memorybank_scores: Dict[str, Any] = {}

for ds in DATASETS:
    path = RESULTS_DIR / f"memorybank_{ds}_250_results.json"
    if not path.exists():
        print(f"[missing] memorybank {ds} — run the evaluation loop first")
        memorybank_scores[ds] = None
        continue
    records = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(records, list):
        records = []
    result = score_results(records, pred_key="predicted_answer", gt_key="ground_truth")
    memorybank_scores[ds] = result
    print(f"  MemoryBank | {ds:15s} | {result['correct']:3d}/{result['total']:3d} = {result['accuracy_pct']:5.1f}%")

---
## 8 — Mem1

**Description:** Mem1 compresses conversational history into a compact, iteratively refined memory representation rather than storing raw turns, trading recall precision for token efficiency.

**Paper / Repo:** https://github.com/mem1-ai/mem1

**Python env:** Python 3.10+ (no `sentence-transformers` required).

**Result format:** List of `{model_answer, ground_truth, category}`.

In [ ]:
# %pip install openai

# ── Mem1 system setup ─────────────────────────────────────────────────────────
MEM1_SRC = os.environ.get("MEM1_SRC", "")  # path to Mem1 clone root

if MEM1_SRC and MEM1_SRC not in sys.path:
    sys.path.insert(0, MEM1_SRC)
    print(f"[ok] Added {MEM1_SRC} to sys.path")
else:
    print("[info] MEM1_SRC not set — assuming Mem1 is already importable")

try:
    import openai
    print(f"[ok] openai {openai.__version__}")
except ImportError:
    print("[error] openai not installed")

In [ ]:
# ── Mem1 evaluation loop ──────────────────────────────────────────────────────
# Mem1 maintains a single compressed memory string that is updated after each
# conversation turn.  For each dataset we:
#   1. Instantiate Mem1.
#   2. Feed all conversation turns to compress into memory.
#   3. Query each question using the compressed memory as context.

def run_mem1_dataset(dataset_name: str, force: bool = False):
    """Run Mem1 evaluation on one dataset. Resumes from saved results."""
    out_path = RESULTS_DIR / f"mem1_{dataset_name}_250_results.json"

    existing: List = []
    if out_path.exists() and not force:
        existing = json.loads(out_path.read_text(encoding="utf-8"))
        if not isinstance(existing, list):
            existing = []
        print(f"[resume] {dataset_name}: {len(existing)} records already saved")

    data       = load_dataset(dataset_name)
    done_count = len(existing)
    remaining  = data[done_count:]

    if not remaining:
        print(f"[skip]   {dataset_name}: already complete ({len(existing)} records)")
        return

    print(f"[run]    {dataset_name}: {len(remaining)} questions remaining …")

    # ── Import Mem1 ───────────────────────────────────────────────────────────
    try:
        from mem1 import Mem1Agent  # type: ignore  # adjust to actual module
    except ImportError:
        print("[error] Mem1 not importable — set MEM1_SRC env var")
        return

    agent = Mem1Agent(openai_api_key=OPENAI_API_KEY)

    # Compress conversation history into Mem1
    for item in data:
        q   = str(item.get("question", ""))
        ans = str(item.get("answer", ""))
        if q.strip():
            agent.update(f"Q: {q}  A: {ans}")

    results = list(existing)

    for i, item in enumerate(remaining, 1):
        question = item["question"]
        gt       = item.get("answer", "")
        category = item.get("category", "")

        try:
            model_answer = agent.answer(question)
        except Exception as e:
            print(f"  [err] q{done_count+i}: {e}")
            model_answer = ""

        results.append({"model_answer": model_answer, "ground_truth": gt, "category": category})

        if i % 10 == 0 or i == len(remaining):
            out_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
            print(f"  [{done_count+i}/{len(data)}] saved")

    print(f"[done]   {dataset_name}: {len(results)} total records → {out_path}")


# Un-comment to run:
# for ds in DATASETS:
#     run_mem1_dataset(ds)

In [ ]:
# ── Mem1: load results & compute scores ───────────────────────────────────────
mem1_scores: Dict[str, Any] = {}

for ds in DATASETS:
    path = RESULTS_DIR / f"mem1_{ds}_250_results.json"
    if not path.exists():
        print(f"[missing] mem1 {ds} — run the evaluation loop first")
        mem1_scores[ds] = None
        continue
    records = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(records, list):
        records = []
    result = score_results(records, pred_key="model_answer", gt_key="ground_truth")
    mem1_scores[ds] = result
    print(f"  Mem1 | {ds:15s} | {result['correct']:3d}/{result['total']:3d} = {result['accuracy_pct']:5.1f}%")

---
## 9 — MemGPT / Letta

**Description:** MemGPT (now Letta) implements a tiered memory OS for LLMs with in-context working memory, recall storage (conversation history), and archival storage (large external databases), managed via self-directed function calls.

**Paper / Repo:** https://github.com/cpacker/MemGPT

**Python env:** Python 3.10+ (no `sentence-transformers` required — safe with pyenv).

**Note:** The basic Letta Cloud tier does not include `archival_memory_search`. For LoCoMo evaluation a local OpenAI function-calling implementation is used; SimpleQA/DeepResearcher can use Letta Cloud directly.

**Result format:** List of `{model_answer, ground_truth, category}`.

In [ ]:
# %pip install letta openai

# ── MemGPT/Letta system setup ─────────────────────────────────────────────────
LETTA_API_KEY = os.environ.get("LETTA_API_KEY", "")  # Letta Cloud API key (optional)
LETTA_BASE_URL = os.environ.get("LETTA_BASE_URL", "https://api.letta.com")  # or local server

try:
    import letta
    print(f"[ok] letta {letta.__version__}")
except ImportError:
    print("[error] letta not installed. Run: pip install letta")

print(f"[info] LETTA_BASE_URL = {LETTA_BASE_URL}")
print(f"[info] LETTA_API_KEY  = {'set' if LETTA_API_KEY else 'not set'}")

In [ ]:
# ── MemGPT evaluation loop ────────────────────────────────────────────────────
# MemGPT manages its own memory OS; we send conversation turns and then query.
#
# For LoCoMo: we iterate all sessions, send each turn to the agent, then ask
# the question.  (Local function-calling implementation avoids Cloud tier limits.)
#
# For SimpleQA/DeepResearcher: stub context is sent, then each question is asked.

def run_memgpt_dataset(dataset_name: str, force: bool = False):
    """Run MemGPT evaluation on one dataset. Resumes from saved results."""
    out_path = RESULTS_DIR / f"memgpt_{dataset_name}_250_results.json"

    existing: List = []
    if out_path.exists() and not force:
        existing = json.loads(out_path.read_text(encoding="utf-8"))
        if not isinstance(existing, list):
            existing = []
        print(f"[resume] {dataset_name}: {len(existing)} records already saved")

    data       = load_dataset(dataset_name)
    done_count = len(existing)
    remaining  = data[done_count:]

    if not remaining:
        print(f"[skip]   {dataset_name}: already complete ({len(existing)} records)")
        return

    print(f"[run]    {dataset_name}: {len(remaining)} questions remaining …")

    # ── Initialise Letta client ───────────────────────────────────────────────
    try:
        from letta import create_client  # type: ignore
    except ImportError:
        print("[error] letta not importable")
        return

    if LETTA_API_KEY:
        client = create_client(base_url=LETTA_BASE_URL, token=LETTA_API_KEY)
    else:
        client = create_client()  # local server on localhost:8283

    agent_name = f"eval_{dataset_name}"
    # Create or reuse agent
    try:
        agent = client.get_agent(agent_name=agent_name)
        print(f"[info]   reusing existing agent: {agent_name}")
    except Exception:
        agent = client.create_agent(name=agent_name)
        print(f"[info]   created new agent: {agent_name}")

    # Feed context
    for item in data:
        txt = str(item.get("answer", "")) or str(item.get("question", ""))
        if txt.strip():
            try:
                client.send_message(agent_id=agent.id, message=txt, role="user")
            except Exception:
                pass

    results = list(existing)

    for i, item in enumerate(remaining, 1):
        question = item["question"]
        gt       = item.get("answer", "")
        category = item.get("category", "")

        try:
            response     = client.send_message(agent_id=agent.id, message=question, role="user")
            model_answer = ""
            for msg in response.messages:
                if hasattr(msg, "text") and msg.text:
                    model_answer = msg.text
                    break
        except Exception as e:
            print(f"  [err] q{done_count+i}: {e}")
            model_answer = ""

        results.append({"model_answer": model_answer, "ground_truth": gt, "category": category})

        if i % 10 == 0 or i == len(remaining):
            out_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
            print(f"  [{done_count+i}/{len(data)}] saved")

    print(f"[done]   {dataset_name}: {len(results)} total records → {out_path}")


# Un-comment to run:
# for ds in DATASETS:
#     run_memgpt_dataset(ds)

In [ ]:
# ── MemGPT: load results & compute scores ─────────────────────────────────────
memgpt_scores: Dict[str, Any] = {}

for ds in DATASETS:
    path = RESULTS_DIR / f"memgpt_{ds}_250_results.json"
    if not path.exists():
        print(f"[missing] memgpt {ds} — run the evaluation loop first")
        memgpt_scores[ds] = None
        continue
    records = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(records, list):
        records = []
    result = score_results(records, pred_key="model_answer", gt_key="ground_truth")
    memgpt_scores[ds] = result
    print(f"  MemGPT | {ds:15s} | {result['correct']:3d}/{result['total']:3d} = {result['accuracy_pct']:5.1f}%")

---
## 10 — Summary Table

Aggregate all scores into a single comparison table.  Missing entries (no result file) are shown as `N/A`.

In [ ]:
# ── Build summary ─────────────────────────────────────────────────────────────

all_system_scores = {
    "Mem0"       : mem0_scores,
    "MemoryOS"   : memoryos_scores,
    "A-Mem"      : amem_scores,
    "Memento"    : memento_scores,
    "THEANINE"   : theanine_scores,
    "MemoryBank" : memorybank_scores,
    "Mem1"       : mem1_scores,
    "MemGPT"     : memgpt_scores,
}

def fmt(score_dict: Optional[Dict], ds: str) -> str:
    if score_dict is None or score_dict.get(ds) is None:
        return "N/A"
    entry = score_dict[ds]
    if entry is None:
        return "N/A"
    return f"{entry['accuracy_pct']:.1f}%"


def avg_pct(score_dict: Optional[Dict]) -> str:
    if score_dict is None:
        return "N/A"
    vals = []
    for ds in DATASETS:
        entry = score_dict.get(ds)
        if entry and entry.get("accuracy_pct") is not None:
            vals.append(entry["accuracy_pct"])
    if not vals:
        return "N/A"
    return f"{sum(vals)/len(vals):.1f}%"


# ── Pretty-print table ─────────────────────────────────────────────────────────
header = f"{'System':<14} {'LoCoMo':>10} {'SimpleQA':>10} {'DeepResearcher':>16} {'Avg':>8}"
sep    = "-" * len(header)

print("\nGroup 1 — Conversational Memory QA (Contains-Match %, 250 questions each)")
print(sep)
print(header)
print(sep)

for system, sd in all_system_scores.items():
    locomo = fmt(sd, "locomo")
    simpleqa = fmt(sd, "simpleqa")
    dr = fmt(sd, "deepresearcher")
    avg = avg_pct(sd)
    print(f"{system:<14} {locomo:>10} {simpleqa:>10} {dr:>16} {avg:>8}")

print(sep)
print("\nNote: LoCoMo cat-5 adversarial questions (answer='') are excluded from all scores.")
print("Note: MemoryOS and A-Mem require Python 3.13 (sentence-transformers + NumPy 2.x compat).")

In [ ]:
# ── Optional: pandas DataFrame + styled output ────────────────────────────────
try:
    import pandas as pd

    rows = []
    for system, sd in all_system_scores.items():
        rows.append({
            "System"        : system,
            "LoCoMo"        : fmt(sd, "locomo"),
            "SimpleQA"      : fmt(sd, "simpleqa"),
            "DeepResearcher": fmt(sd, "deepresearcher"),
            "Avg"           : avg_pct(sd),
        })

    df = pd.DataFrame(rows).set_index("System")
    print("\nDataFrame view:")
    display(df)  # type: ignore
except ImportError:
    print("[info] pandas not installed — skipping DataFrame view")